# Решения: venv и README

**Для преподавателя.** Эталон к `lesson.ipynb` и `homework.ipynb`. Не показывать ученикам до сдачи.

In [ ]:
import json
import subprocess
import sys
from pathlib import Path
from importlib.metadata import PackageNotFoundError, version


In [ ]:
venv_commands = [
    'python -m venv .venv',
    '.venv\\Scripts\\activate  # Windows',
    'python -m pip install -U pip',
    'python -m pip install -r requirements.txt',
]

def pkg_line(pkg_name: str, alias: str | None = None) -> str:
    try:
        return f'{alias or pkg_name}=={version(pkg_name)}'
    except PackageNotFoundError:
        return f'{alias or pkg_name}>=0'


requirements_text = '\n'.join(
    [
        pkg_line('numpy'),
        pkg_line('pandas'),
        pkg_line('scikit-learn'),
    ]
) + '\n'
cli_script = Path('../04_practice_metrics_cli/train_cli.py')
data_path = Path('../../data/bank_marketing_slim.csv')
cmd = [sys.executable, str(cli_script), '--data', str(data_path), '--threshold', '0.45']
proc = subprocess.run(cmd, capture_output=True, text=True, check=False)
if proc.returncode != 0:
    raise RuntimeError(proc.stderr or proc.stdout)
metrics = json.loads(proc.stdout)
exp_readme = (
    '# Эксперимент: отклик на депозит (LogisticRegression)\n\n'
    '## Цель\n'
    'Спрогнозировать `y` (yes/no) на срезе UCI Bank Marketing и выбрать рабочий порог.\n\n'
    '## Leakage rule\n'
    'Столбец `duration` исключён из признаков (`assert duration not in features`).\n\n'
    '## Запуск\n'
    '1. Создать venv.\n'
    '2. Установить зависимости из requirements.txt.\n'
    '3. Запустить CLI: `python train_cli.py --data ../../data/bank_marketing_slim.csv --threshold 0.45`.\n\n'
    '## Результаты test\n'
    f"- threshold: {metrics['threshold']:.2f}\\n"
    f"- accuracy: {metrics['accuracy']:.3f}\\n"
    f"- precision: {metrics['precision']:.3f}\\n"
    f"- recall: {metrics['recall']:.3f}\\n"
    f"- f1: {metrics['f1']:.3f}\\n\n"
    '## Ограничения\n'
    'Это учебный slim-срез; порог и метрики требуют пересчёта на полном контуре данных.'
)
tree_text = (
    'submission/\n'
    '  train_cli.py\n'
    '  requirements.txt\n'
    '  README.md\n'
    '  metrics.json\n'
)
RISKS = (
    'Без зафиксированных версий пакетов и явной инструкции по запуску коллега может получить другие метрики. '
    'Второй риск - случайно добавить duration в признаки и получить нереалистично хороший результат.'
)
print('\n'.join(venv_commands))
print(requirements_text)
print(exp_readme[:500])
print(tree_text)
print(RISKS)